## preliminary random forest

In [1]:
# Cell: Load Final Dataset and Define Classification Task

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("PRELIMINARY RANDOM FOREST CLASSIFICATION")
print("="*60)

# Load the final dataset
df = pd.read_csv('/Users/zdiener/data-center-classification/data/processed/datacenters_with_ejscreen_final.csv')

print(f"\nDataset loaded: {len(df):,} records")

# Define our classification task
print("\n" + "="*60)
print("DEFINING CLASSIFICATION TARGET")
print("="*60)

# We want to predict: Is this data center in an environmentally disadvantaged area?
# We'll use multiple criteria to define "disadvantaged"

# Check available EJ indicators
print("\nAvailable EJ indicators for defining target:")

# Option 1: Use EJ Index (combines environmental + demographics)
ej_index_cols = [col for col in df.columns if col.startswith('P_') and col.endswith('_D2')]
print(f"  EJ Index columns: {ej_index_cols}")

# Option 2: Use high environmental burden (>75th percentile on multiple indicators)
env_percentiles = ['P_DSLPM', 'P_PM25', 'P_OZONE', 'P_PTRAF', 'P_PWDIS']
print(f"  Environmental percentiles: {env_percentiles}")

# Option 3: Combine demographic + environmental
print(f"  Minority %: {'MINORPCT' in df.columns}")
print(f"  Low income %: {'LOWINCPCT' in df.columns}")

# Let's create multiple target definitions
print("\n" + "="*60)
print("CREATING TARGET VARIABLE")
print("="*60)

# Definition: Environmentally burdened = High on at least 3 of 5 key indicators (>75th percentile)
key_indicators = ['P_DSLPM', 'P_PM25', 'P_OZONE', 'P_PTRAF', 'P_PWDIS']

# Count how many indicators are >75th percentile for each facility
df['high_burden_count'] = 0
for indicator in key_indicators:
    if indicator in df.columns:
        df['high_burden_count'] += (df[indicator] > 75).astype(int)

# Target: 1 if high burden on 3+ indicators, 0 otherwise
df['environmentally_disadvantaged'] = (df['high_burden_count'] >= 3).astype(int)

# Alternative target: EJ Index approach (if available)
if 'P_DSLPM_D2' in df.columns:
    df['ej_disadvantaged'] = (df['P_DSLPM_D2'] > 75).astype(int)

print(f"\nTarget variable created: 'environmentally_disadvantaged'")
print(f"  Definition: High burden (>75th percentile) on 3+ of 5 key indicators")
print(f"  Key indicators: {key_indicators}")

print(f"\nTarget distribution:")
print(df['environmentally_disadvantaged'].value_counts())
print(f"\nPercentages:")
print(df['environmentally_disadvantaged'].value_counts(normalize=True) * 100)

# Check class imbalance
class_counts = df['environmentally_disadvantaged'].value_counts()
if len(class_counts) == 2:
    minority_class = class_counts.min()
    majority_class = class_counts.max()
    imbalance_ratio = majority_class / minority_class
    print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}:1")
    if imbalance_ratio > 3:
        print("⚠️  Significant class imbalance detected - will use balanced approach")

PRELIMINARY RANDOM FOREST CLASSIFICATION

Dataset loaded: 2,536 records

DEFINING CLASSIFICATION TARGET

Available EJ indicators for defining target:
  EJ Index columns: []
  Environmental percentiles: ['P_DSLPM', 'P_PM25', 'P_OZONE', 'P_PTRAF', 'P_PWDIS']
  Minority %: False
  Low income %: True

CREATING TARGET VARIABLE

Target variable created: 'environmentally_disadvantaged'
  Definition: High burden (>75th percentile) on 3+ of 5 key indicators
  Key indicators: ['P_DSLPM', 'P_PM25', 'P_OZONE', 'P_PTRAF', 'P_PWDIS']

Target distribution:
environmentally_disadvantaged
0    1956
1     580
Name: count, dtype: int64

Percentages:
environmentally_disadvantaged
0    77.129338
1    22.870662
Name: proportion, dtype: float64

Class imbalance ratio: 3.37:1
⚠️  Significant class imbalance detected - will use balanced approach


In [2]:
# Cell: Feature Engineering and Selection

print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Select features for classification
# We'll use: facility characteristics + location + some environmental context

feature_categories = {
    'facility_characteristics': [
        'frac_mw_numeric',              # Power capacity
        'facility_size_sqft',            # Size (if available from OSDC)
        'sqft',                          # Size from OSDC
        'is_operational',                # Status
        'is_proposed',                   # Status
        'is_campus',                     # Type
        'is_building',                   # Type
    ],
    'location': [
        'latitude',
        'longitude',
        'state_density',                 # From our matching (sparse/dense/etc)
    ],
    'demographics': [
        'ACSTOTPOP',                     # Total population
        'MINORPCT',                      # Percent minority
        'LOWINCPCT',                     # Percent low income
        'OVER64',                        # Population over 64
    ],
    'environmental_raw': [
        'DSLPM',                         # Diesel particulate matter
        'PM25',                          # PM 2.5
        'OZONE',                         # Ozone
        'PTRAF',                         # Traffic proximity
        'PWDIS',                         # Wastewater discharge
        'PNPL',                          # Proximity to Superfund
        'PRMP',                          # Proximity to RMP facilities
        'PTSDF',                         # Proximity to hazardous waste
        'PRE1960',                       # Lead paint indicator
        'UST',                           # Underground storage tanks
    ]
}

# Collect available features
available_features = []
for category, features in feature_categories.items():
    for feature in features:
        if feature in df.columns:
            available_features.append(feature)

print(f"\nAvailable features by category:")
for category, features in feature_categories.items():
    available = [f for f in features if f in df.columns]
    print(f"  {category}: {len(available)}/{len(features)} available")
    if len(available) < len(features):
        missing = [f for f in features if f not in df.columns]
        print(f"    Missing: {missing}")

print(f"\nTotal available features: {len(available_features)}")

# Create feature matrix
print(f"\n" + "="*60)
print("CREATING FEATURE MATRIX")
print(f"="*60)

# Filter to records with target variable and at least some features
df_model = df[df['environmentally_disadvantaged'].notna()].copy()

print(f"\nRecords with target variable: {len(df_model):,}")

# Handle missing values in features
print(f"\nHandling missing values...")

# Check missingness
for feature in available_features:
    missing_pct = df_model[feature].isna().sum() / len(df_model) * 100
    if missing_pct > 50:
        print(f"  ⚠️  {feature}: {missing_pct:.1f}% missing - excluding")
        available_features.remove(feature)
    elif missing_pct > 0:
        print(f"  {feature}: {missing_pct:.1f}% missing - will impute")

# Create feature matrix
X = df_model[available_features].copy()
y = df_model['environmentally_disadvantaged'].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Impute missing values with median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print(f"\n✓ Missing values imputed with median")

# Check for any remaining issues
print(f"\nFinal data quality check:")
print(f"  Infinite values: {np.isinf(X_imputed.values).sum()}")
print(f"  NaN values: {X_imputed.isna().sum().sum()}")

# Remove any infinite values
X_imputed = X_imputed.replace([np.inf, -np.inf], np.nan)
X_imputed = X_imputed.fillna(X_imputed.median())

print(f"\n✓ Feature matrix ready for modeling")

FEATURE ENGINEERING

Available features by category:
  facility_characteristics: 6/7 available
    Missing: ['is_proposed']
  location: 3/3 available
  demographics: 3/4 available
    Missing: ['MINORPCT']
  environmental_raw: 10/10 available

Total available features: 22

CREATING FEATURE MATRIX

Records with target variable: 2,536

Handling missing values...
  ⚠️  frac_mw_numeric: 77.0% missing - excluding
  ⚠️  sqft: 69.6% missing - excluding
  ⚠️  is_campus: 51.0% missing - excluding
  ⚠️  state_density: 51.0% missing - excluding
  LOWINCPCT: 0.4% missing - will impute
  OVER64: 0.4% missing - will impute
  DSLPM: 0.7% missing - will impute
  PM25: 0.5% missing - will impute
  OZONE: 0.5% missing - will impute
  PTRAF: 0.7% missing - will impute
  PWDIS: 0.7% missing - will impute
  PNPL: 0.7% missing - will impute
  PRMP: 0.7% missing - will impute
  PTSDF: 0.7% missing - will impute
  PRE1960: 0.4% missing - will impute
  UST: 0.4% missing - will impute

Feature matrix shape: (25